In [ ]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# 1. Fetch Transcript
# Replace with your desired YouTube URL
video_url = "https://www.youtube.com/watch?v=L_Guz73e6fw" 
loader = YoutubeLoader.from_youtube_url(video_url, add_video_info=True)
transcript_docs = loader.load()

print(f"Loaded transcript from: {transcript_docs[0].metadata['title']}")

In [ ]:
# 2. Document Splitting (as requested)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(transcript_docs)

print(f"Split transcript into {len(chunks)} chunks.")

In [ ]:
# 3. Embeddings & Vector Store
# Using HuggingFace since HF token is present in .env
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

print("Vector store created successfully.")

In [ ]:
# 4. Query Engine (RAG Chain)
model = ChatOpenAI(
    model="meta-llama/Llama-3.1-8B-Instruct",
    openai_api_key=os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN"),
    openai_api_base="https://router.huggingface.co/v1",
)

qa_chain = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

query = "What is the main topic of this video?"
response = qa_chain.invoke(query)

print(f"Query: {query}")
print(f"Response: {response['result']}")